In [ ]:
# ============================================================
# 1. IMPORTS
# ============================================================

import json
import numpy as np
import random
import math
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

print("📌 Imports loaded.")

In [ ]:
# ============================================================
# 2. FEATURES USED BY THE MODEL
# ============================================================

FEATURES = [
    "lat", "lon", "altitude_ft",
    "gspeed", "vspeed", "track",
    "edr_300hPa_m23s1",
    "turbulence_ellrod_300hPa_s2",
    "richardson_number_300hPa_idx",
    "turbulence_cape_m23s",
    "wind_speed_10000m_ms", "wind_dir_10000m_d",
    "wind_speed_2m_ms", "wind_dir_2m_d",
    "minutes_since_start"
]

print("📌 Using features:", FEATURES)


In [ ]:
# ============================================================
# 3. LOAD DATASET
# ============================================================

print("📥 Loading dataset...")
with open("../data/CDG-FCO/flight_data_with_minutes_since_start.json", "r") as f:
    flights_data = json.load(f)
print(f"📌 Loaded {len(flights_data)} flights.")


In [ ]:
# ============================================================
# 4. EXTRACT PHASE SEQUENCES
# ============================================================

def extract_phase_sequences(flights_data, phase):
    seqs = []
    for flight in flights_data:
        fid = flight["flight_metadata"]["fr24_id"]  # flight id
        if phase in flight:
            seq = flight[phase]
            seqs.append((seq, {"flight_id": fid, "phase": phase}))
    return seqs

In [ ]:
takeoff_sequences  = extract_phase_sequences(flights_data, "take_off")
cruise_sequences   = extract_phase_sequences(flights_data, "cruising")
landing_sequences  = extract_phase_sequences(flights_data, "landing")

HERE WE WILL EXTRACT 100 FLIGHT ID. THOSE ARE THE FLIGHTS USED TO CALCULATE THE AVERAGE HAVERSINE ERROR OF OUR LSTM MODEL

In [ ]:
import random
import json

# ==========================================================
# STEP 1 — Extract all flight IDs
# ==========================================================
all_flight_ids = list({meta['flight_id'] for _, meta in takeoff_sequences})
print(f"Total unique flights: {len(all_flight_ids)}")

# ==========================================================
# STEP 2 — Randomly sample 100 flights for validation
# ==========================================================
random.seed(42)  # for reproducibility
validation_flights = random.sample(all_flight_ids, 100)

# ==========================================================
# STEP 3 — Save to JSON
# ==========================================================
output_file = "lstm_validation_flight_ids.json"

with open(output_file, "w", encoding="utf-8") as f:
    json.dump(validation_flights, f, indent=4, ensure_ascii=False)

print(f"✅ Saved {len(validation_flights)} flight IDs to {output_file}")


In [ ]:
def remove_experiment_flights(sequences, EXCLUDED_IDS):
    clean = []
    for seq, meta in sequences:
        if meta["flight_id"] not in EXCLUDED_IDS:
            clean.append((seq, meta))
    return clean

In [ ]:
import json

# Path to your JSON file
input_file = "lstm_validation_flight_ids.json"

with open(input_file, "r", encoding="utf-8") as f:
    validation_flight_ids = json.load(f)

print(f"Loaded {len(validation_flight_ids)} flight IDs from {input_file}")

takeoff_sequences = remove_experiment_flights(takeoff_sequences, validation_flight_ids)
cruise_sequences  = remove_experiment_flights(cruise_sequences, validation_flight_ids)
landing_sequences = remove_experiment_flights(landing_sequences, validation_flight_ids)

print(len(takeoff_sequences))
print(len(cruise_sequences))
print(len(landing_sequences))

In [ ]:
def seq_to_matrix(seq):
    return np.array([[p.get(f, 0) for f in FEATURES] for p in seq], dtype=np.float32)


In [ ]:
# ============================================================
# 6. NORMALIZATION (TRAIN ONLY)
# ============================================================

def compute_stats(sequences):
    mats = [seq_to_matrix(seq) for seq in sequences]
    all_pts = np.vstack(mats)
    means = all_pts.mean(axis=0)
    stds  = all_pts.std(axis=0) + 1e-6
    return means, stds



In [ ]:
# ============================================================
# 7. VARIABLE-WINDOW DATASET
# ============================================================

class PhaseDataset(Dataset):
    def __init__(self, sequences, means, stds, min_w=2, max_w=30):
        self.means, self.stds = means, stds
        self.samples = []

        print("📦 Building samples...")

        for seq in sequences:
            mat = (seq_to_matrix(seq) - means) / stds
            T = len(mat)
            for i in range(T - min_w - 1):
                w = random.randint(min_w, max_w)
                if i + w >= T:
                    continue
                self.samples.append((mat[i:i+w], mat[i+w]))

        print(f"📌 Total samples: {len(self.samples)}")

    def __len__(self): return len(self.samples)
    def __getitem__(self, idx):
        x, y = self.samples[idx]
        return torch.tensor(x, dtype=torch.float32), torch.tensor(y, dtype=torch.float32)


In [ ]:
def collate_fn(batch):
    xs, ys = zip(*batch)
    lengths = [len(x) for x in xs]
    max_len = max(lengths)

    padded = []
    for x in xs:
        pad_len = max_len - len(x)
        padded.append(torch.cat([x, torch.zeros(pad_len, x.shape[1])], dim=0))

    return torch.stack(padded), torch.stack(ys), torch.tensor(lengths)



In [ ]:
# ============================================================
# 9. LSTM BASELINE MODEL
# ============================================================

class BaselineLSTM(nn.Module):
    def __init__(self, input_dim, hidden=128, layers=6, bidirectional=False):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, hidden, layers, batch_first=True, bidirectional=bidirectional)
        self.fc   = nn.Linear(hidden, input_dim)

    def forward(self, x, lengths):
        packed = nn.utils.rnn.pack_padded_sequence(
            x, lengths.cpu(), batch_first=True, enforce_sorted=False
        )
        _, (h, _) = self.lstm(packed)
        last = h[-1]
        return self.fc(last)

In [ ]:
# ============================================================
# 10. TRAIN A PHASE MODEL
# ============================================================

def train_phase_model(train_sequences, means, stds, epochs=50, bidirectional=False):
    dataset = PhaseDataset(train_sequences, means, stds)
    loader  = DataLoader(dataset, batch_size=64, shuffle=True, collate_fn=collate_fn)

    model = BaselineLSTM(input_dim=len(FEATURES), bidirectional=bidirectional)
    opt   = torch.optim.Adam(model.parameters(), lr=1e-2)
    loss_fn = nn.MSELoss()

    print("🚀 Training...")
    for ep in range(1, epochs+1):
        total = 0
        for xb, yb, lengths in loader:
            pred = model(xb, lengths)
            loss = loss_fn(pred, yb)

            opt.zero_grad()
            loss.backward()
            opt.step()

            total += loss.item()

        print(f"📘 Epoch {ep:02d} | Loss {total/len(loader):.6f}")

    print("✅ Training complete.")
    return model

In [ ]:
# ============================================================
# 11. AUTOREGRESSIVE PREDICTION
# ============================================================
def predict_future(model, history_raw, means, stds, steps):
    history = (history_raw - means) / stds
    x = torch.tensor(history).unsqueeze(0)
    lengths = torch.tensor([len(history)])

    preds = []

    for _ in range(steps):
        pred_norm = model(x, lengths).detach().numpy()[0]
        preds.append(pred_norm)

        pred_tensor = torch.tensor(pred_norm).unsqueeze(0).unsqueeze(0)
        x = torch.cat([x[:,1:], pred_tensor], dim=1) if x.shape[1] > 1 else pred_tensor.clone()
        lengths = torch.tensor([x.shape[1]])

    preds = np.array(preds)
    return preds * stds + means


In [ ]:
# ============================================================
# 12. HAVERSINE DISTANCE (meters)
# ============================================================

def haversine(lat1, lon1, lat2, lon2):
    R = 6371e3
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlambda = math.radians(lon2 - lon1)
    a = math.sin(dphi/2)**2 + math.cos(phi1)*math.cos(phi2)*math.sin(dlambda/2)**2
    return R * 2 * math.atan2(math.sqrt(a), math.sqrt(1-a))



In [ ]:
# ============================================================
# 13. MODEL EVALUATION ON TEST SET
# ============================================================

def evaluate_phase(model, test_sequences, means, stds):
    errors = []

    for seq in test_sequences:
        mat = seq_to_matrix(seq)
        T = len(mat)
        mid = T // 2

        history = mat[:mid]
        target  = mat[mid:]

        try:
            preds = predict_future(model, history, means, stds, steps=len(target))

            # error over lat/lon
            e = np.mean([
                haversine(t[0], t[1], p[0], p[1])
                for t, p in zip(target, preds)
            ])
            errors.append(e)
        except:
            continue

    print(f"📌 Avg Error = {np.mean(errors):.2f} meters")
    return errors



In [ ]:
def train_and_eval_phase(sequences, phase_name, epochs=50, bidirectional = False):
    print("\n\n==============================")
    print(f"🔵 PHASE: {phase_name.upper()}")
    print("==============================")
    sequences = [seq for (seq, meta) in sequences]
    train_seqs, test_seqs = train_test_split(sequences, test_size=0.1, random_state=42)

    means, stds = compute_stats(train_seqs)
    print("📊 Normalization computed.")

    model = train_phase_model(
        train_sequences=train_seqs, 
        means=means, 
        stds=stds,
        epochs=epochs, 
        bidirectional=bidirectional
        )
    
    print("📊 Evaluating...")
    errors = evaluate_phase(model, test_seqs, means, stds)

    return model, means, stds, errors


In [ ]:
model_takeoff, means_takeoff, stds_takeoff, errors_takeoff = train_and_eval_phase(takeoff_sequences, "takeoff")
model_cruise,  means_cruise,  stds_cruise,  errors_cruise  = train_and_eval_phase(cruise_sequences, "cruising")
model_landing, means_landing, stds_landing, errors_landing = train_and_eval_phase(landing_sequences, "landing")

LETS TRAIN THE BI DIRECTIONAL LSTM MODEL 

In [ ]:
model_takeoff, means_takeoff, stds_takeoff, errors_takeoff = train_and_eval_phase(
    sequences=takeoff_sequences, 
    phase_name="takeoff", 
    epochs=100
    )

In [ ]:
model_cruise,  means_cruise,  stds_cruise,  errors_cruise  = train_and_eval_phase(
    sequences=cruise_sequences, 
    phase_name="cruising", 
    epochs=100
    )

In [ ]:
model_landing, means_landing, stds_landing, errors_landing = train_and_eval_phase(
    sequences=landing_sequences, 
    phase_name="landing", 
    epochs=100
    )

In [ ]:
# ============================================================
#   PREDICTION EXAMPLE FOR TAKEOFF, CRUISE, LANDING
# ============================================================
import os
import json

def save_lstm_results(json_path, phase_name, observed, ground_truth, predicted, mean_error):
    """
    Stores LSTM predictions for a given phase inside a JSON file.

    json_path   : path to JSON file
    phase_name  : "takeoff", "cruising", "landing"
    observed    : numpy array of observed points
    ground_truth: numpy array of GT points
    predicted   : numpy array of predicted points
    mean_error  : float
    """

    # Load existing JSON file if present
    if os.path.exists(json_path):
        with open(json_path, "r") as f:
            results = json.load(f)
    else:
        results = {}

    # Insert or update section for the phase
    results[phase_name] = {
        "observed": observed,
        "ground_truth": ground_truth,
        "predicted": predicted,
        "mean_haversine_error": float(mean_error)
    }

    # Save back to disk
    with open(json_path, "w") as f:
        json.dump(results, f, indent=4)

    print(f"📁 Saved results for phase '{phase_name}' into {json_path}")


def predict_positions(example_seq, model, means, stds):
    # Convert to numeric matrix
    mat = seq_to_matrix(example_seq)
    T = len(mat)
    mid = T // 2

    # First half = history
    history_raw = mat[:mid]
    # Second half = ground truth to predict
    gt_future = mat[mid:]

    # Predict same number of steps
    preds = predict_future(model, history_raw, means, stds, steps=len(gt_future))

    for i in range(len(gt_future)):
        preds[i][-1] = gt_future[i][-1]

    # Compute error
    errors = [
        haversine(gt_future[i][0], gt_future[i][1],
                  preds[i][0], preds[i][1])
        for i in range(len(preds))
    ]
    # print(f"📌 Mean Haversine Error: {np.mean(errors):.2f} meters\n")

    # ============================================================
    #   SAVE RESULTS USING THE HELPER FUNCTION
    # ============================================================
    history_raw = history_raw.astype(float).tolist()
    gt_future = gt_future.astype(float).tolist()
    preds = preds.astype(float).tolist()
    return history_raw, gt_future, preds, np.mean(errors)


def plot_phase_prediction(example_seq, model, means, stds, phase_name, json_path="results_LSTM.json"):
    """
    example_seq: the raw sequence dict list for the phase
    model     : trained LSTM model for that phase
    means,stds: normalization stats for the phase
    """
    print(f"\n==============================")
    print(f"📌 Prediction Example — {phase_name.upper()}")
    print("==============================")

    # Convert to numeric matrix
    mat = seq_to_matrix(example_seq)
    T = len(mat)
    mid = T // 2

    # First half = history
    history_raw = mat[:mid]
    # Second half = ground truth to predict
    gt_future = mat[mid:]

    # Predict same number of steps
    preds = predict_future(model, history_raw, means, stds, steps=len(gt_future))

    for i in range(len(gt_future)):
        preds[i][-1] = gt_future[i][-1]

    # Compute error
    errors = [
        haversine(gt_future[i][0], gt_future[i][1],
                  preds[i][0], preds[i][1])
        for i in range(len(preds))
    ]
    print(f"📌 Mean Haversine Error: {np.mean(errors):.2f} meters\n")

    # ============================================================
    #   SAVE RESULTS USING THE HELPER FUNCTION
    # ============================================================
    history_raw, gt_future, preds, mean_error = predict_positions(example_seq, model, means, stds)

    save_lstm_results(
        json_path=json_path,
        phase_name=phase_name,
        observed=[[p[0], p[1], p[-1]] for p in history_raw],
        ground_truth=[[p[0], p[1], p[-1]] for p in gt_future],
        predicted=[[p[0], p[1], p[-1]] for p in preds],
        mean_error=mean_error
    )


    # -------- Plot results --------
    plt.figure(figsize=(8, 6))

    # Plot history
    plt.scatter(history_raw[:, 1], history_raw[:, 0],
                color="blue", label="Observed (first half)", s=20)

    # Plot ground truth future
    plt.plot(gt_future[:, 1], gt_future[:, 0],
             color="green", label="Ground Truth (second half)", linewidth=2)

    # Plot predicted trajectory
    plt.plot(preds[:, 1], preds[:, 0],
             color="red", label="Predicted Future", linewidth=2)

    plt.xlabel("Longitude")
    plt.ylabel("Latitude")
    plt.title(f"Prediction Example — {phase_name.capitalize()}")
    plt.legend()
    plt.grid(True)
    plt.show()


In [ ]:
def get_phase_sequence(flights_data, flight_id, phase):
    for flight in flights_data:
        if flight["flight_metadata"]["fr24_id"] == flight_id:
            return flight.get(phase, None)
    return None

In [ ]:
# ============================================================
#   RUN EXAMPLES FOR THE 3 PHASES
# ============================================================

# Random example from each phase test set
example_takeoff = get_phase_sequence(flights_data, "39ba2570", "take_off")
example_cruising = get_phase_sequence(flights_data, "39ba2570", "cruising")
example_landing = get_phase_sequence(flights_data, "39ba2570", "landing")

# plot_phase_prediction(example_takeoff, model_takeoff, means_takeoff, stds_takeoff, "takeoff", json_path="data/results_BILSTM.json")
# plot_phase_prediction(example_cruising,  model_cruise,  means_cruise,  stds_cruise,  "cruising", json_path="data/results_BILSTM.json")
# plot_phase_prediction(example_landing, model_landing, means_landing, stds_landing, "landing", json_path="data/results_BILSTM.json")

NOW LETS PREDICT ALL THE MISSING POSITIONS FOR THE 3 PHASES OF OUR VALIDATION SET

In [ ]:
import json

# Path to your JSON file
input_file = "../data/CDG-FCO/RESULTS/test_sample.json"

with open(input_file, "r", encoding="utf-8") as f:
    test_sample = json.load(f)

def get_mean_haversine_error(
    test_sample, 
    flights_data, 
    model_takeoff, 
    means_takeoff, 
    stds_takeoff,
    model_cruise, 
    means_cruise, 
    stds_cruise,
    model_landing, 
    means_landing, 
    stds_landing
    ):
    results = {"LSTM": {"take_off": 0, "cruising": 0, "landing": 0}}

    takeoff_errors = []
    cruising_errors = []
    landing_errors = []

    take_off_sample = test_sample["take_off"]
    cruising_sample = test_sample["cruising"]
    landing_sample = test_sample["landing"]

    for flight_id in take_off_sample:
        seq = get_phase_sequence(flights_data, flight_id, "take_off")
        pred = predict_positions(seq, model_takeoff, means_takeoff, stds_takeoff)[-1]
        takeoff_errors.append(round(pred/1000, 2))
    
    for flight_id in cruising_sample:
        seq = get_phase_sequence(flights_data, flight_id, "cruising")
        pred = predict_positions(seq, model_cruise, means_cruise, stds_cruise)[-1]
        cruising_errors.append(round(pred/1000, 2))

    for flight_id in landing_sample:
        seq = get_phase_sequence(flights_data, flight_id, "landing")
        pred = predict_positions(seq, model_landing, means_landing, stds_landing)[-1]
        landing_errors.append(round(pred/1000, 2))

    results["LSTM"]["take_off"] = takeoff_errors
    results["LSTM"]["cruising"] = cruising_errors
    results["LSTM"]["landing"] = landing_errors

    return results


In [ ]:
results = get_mean_haversine_error(test_sample, flights_data, model_takeoff, means_takeoff, stds_takeoff, model_cruise, means_cruise, stds_cruise, model_landing, means_landing, stds_landing)

In [ ]:
results

In [ ]:
import json

output_file = "../data/CDG-FCO/RESULTS/MEAN_HAVERSINE_LSTM.json"

with open(output_file, "w", encoding="utf-8") as f:
    json.dump(results, f, indent=4, ensure_ascii=False)

print(f"✅ Saved to {output_file}")


WHAT WE WILL DO KNOW IS TO TRAIN OUR MODEL ON MORE AND MORE SAMPLES (50 flights, then 100 flights, 150, 200 etc until 831 flight which is all my dataset without the lstm_validation_flight_ids.json) AND COMPUTE THE MEAN HAVERSINE ERROR FOR THE 3 DIFFERENT FLIGHT PHASES. FOR EACH BATCH WE WILL SAVE THE LIST OF FLIGHT IDS OF THE BATCH SO THAT WE CAN RUN THE RAG ON THEM (the retrieval phase is limited to the flights present in the batch) AND ALSO COMPUTE THE MEAN HAVERSINE ERROR OF THE RAG.
THEN WE WILL COMPUTE 3 GRAPH WITH X = NUMBER OF FLIGHT IN SAMPLE Y = MEAN HAVERSINE (OF RAG AND LSTM)
ONE GRAPH PER FLIGHT DATA

WHAT WE PLAN TO SEE IS THAT THE RAG CONVERGE TO A LOWER MSE WAY FASTER THAN LSTM

WE NEED TO RUN THE TEST ON THE lstm_validation_flight_ids.json file (100 flights that are already taken out of the training and test set)


In [ ]:
import json

print("📥 Loading dataset...")
with open("data/CDG-FCO/flight_data_with_minutes_since_start.json", "r") as f:
    flights_data = json.load(f)
print(f"📌 Loaded {len(flights_data)} flights.")

takeoff_sequences  = extract_phase_sequences(flights_data, "take_off")
cruise_sequences   = extract_phase_sequences(flights_data, "cruising")
landing_sequences  = extract_phase_sequences(flights_data, "landing")

# Path to your JSON file
input_file = "data/CDG-FCO/lstm_validation_flight_ids.json"

with open(input_file, "r", encoding="utf-8") as f:
    validation_flight_ids = json.load(f)

print(f"Loaded {len(validation_flight_ids)} flight IDs from {input_file}")

takeoff_sequences = remove_experiment_flights(takeoff_sequences, validation_flight_ids)
cruise_sequences  = remove_experiment_flights(cruise_sequences, validation_flight_ids)
landing_sequences = remove_experiment_flights(landing_sequences, validation_flight_ids)

print(len(takeoff_sequences))
print(len(cruise_sequences))
print(len(landing_sequences))

In [ ]:
import os
import json
import random

# =========================
# 1) Extract pool IDs (831)
# =========================
pool_ids = [x[1]["flight_id"] for x in takeoff_sequences]

# remove duplicates while preserving order (just in case)
pool_ids = list(dict.fromkeys(pool_ids))

assert len(pool_ids) == 831, f"Expected 831 pool ids, got {len(pool_ids)}"

# Optional: verify cruise/landing contain the same ID set
cruise_ids  = set(x[1]["flight_id"] for x in cruise_sequences)
landing_ids = set(x[1]["flight_id"] for x in landing_sequences)
assert set(pool_ids) == cruise_ids == landing_ids, "Phase pools do not match!"

print(f"✅ Pool flight IDs extracted: {len(pool_ids)}")

# =========================
# 2) Nested batches
# =========================
SEED = 42
rng = random.Random(SEED)

shuffled_ids = pool_ids[:]
rng.shuffle(shuffled_ids)

batch_sizes = [50, 100, 150, 200, 250, 300, 350, 400, 450, 500, 550, 600, 650, 700, 750, 800, 831]  # edit if you want

# =========================
# 3) Save batch files
# =========================
out_dir = "data/EXPERIMENTS/MSE_CONVERGENCE_SPEED/batches"
os.makedirs(out_dir, exist_ok=True)

# save meta + full shuffled pool for reproducibility
with open(os.path.join(out_dir, "meta.json"), "w", encoding="utf-8") as f:
    json.dump(
        {"seed": SEED, "pool_size": len(shuffled_ids), "batch_sizes": batch_sizes},
        f,
        ensure_ascii=False,
        indent=2
    )

with open(os.path.join(out_dir, "pool_ids_shuffled.json"), "w", encoding="utf-8") as f:
    json.dump(shuffled_ids, f, ensure_ascii=False, indent=2)

# save each nested prefix
for s in batch_sizes:
    batch_ids = shuffled_ids[:s]
    out_path = os.path.join(out_dir, f"batch_ids_{s}.json")
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(batch_ids, f, ensure_ascii=False, indent=2)

print(f"✅ Saved {len(batch_sizes)} batch files in: {out_dir}")


In [ ]:
import os
import json
from pathlib import Path

# =========================
# CONFIG
# =========================
BATCH_DIR = Path("data/EXPERIMENTS/MSE_CONVERGENCE_SPEED/batches")
RESULTS_PATH = Path("experiments/sample_efficiency/results_LSTM.json")

PHASES = ["take_off", "cruising", "landing"]


# =========================
# HELPERS
# =========================
def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def save_json(path, obj):
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)

def safe_read_results(path: Path):
    if not path.exists():
        return {"meta": {}, "results": []}
    return load_json(path)

def upsert_result_record(results_obj, sample_size: int, payload: dict):
    recs = results_obj.setdefault("results", [])
    for rec in recs:
        if int(rec.get("sample_size")) == int(sample_size):
            rec.update(payload)
            return
    recs.append({"sample_size": sample_size, **payload})
    recs.sort(key=lambda x: int(x["sample_size"]))

def filter_sequences_by_ids(sequences, allowed_ids_set):
    # sequences elements have x["flight_id"]
    return [x for x in sequences if x[1]["flight_id"] in allowed_ids_set]


# =========================
# YOUR EXISTING FUNCTION (unchanged)
# BUT it needs access to the trained models/means/stds.
# We’ll define it *inside* the loop so it captures the right models.
# =========================

def run_lstm_loop_over_batches(
    flights_data,
    takeoff_sequences,
    cruise_sequences,
    landing_sequences,
    validation_flight_ids,
):
    # Read meta to get batch sizes (or you can hardcode your list)
    meta_path = BATCH_DIR / "meta.json"
    meta = load_json(meta_path)

    batch_sizes = meta["batch_sizes"]

    results_obj = safe_read_results(RESULTS_PATH)
    results_obj["meta"] = {
        "batch_dir": str(BATCH_DIR.as_posix()),
        "data_source": "data/CDG-FCO/flight_data_with_minutes_since_start.json",
        "val_ids_file": "data/CDG-FCO/lstm_validation_flight_ids.json",
        "batch_sizes": batch_sizes,
        "phases": PHASES,
    }

    for S in batch_sizes:
        batch_file = BATCH_DIR / f"batch_ids_{S}.json"
        batch_ids = load_json(batch_file)
        allowed_ids_set = set(batch_ids)

        print(f"\n==============================")
        print(f"🚀 Sample size = {S}")
        print(f"📌 Loading batch ids from: {batch_file}")
        print(f"==============================")

        # 1) extract the X flights from the sequences
        takeoff_sequences_S = filter_sequences_by_ids(takeoff_sequences, allowed_ids_set)
        cruise_sequences_S  = filter_sequences_by_ids(cruise_sequences, allowed_ids_set)
        landing_sequences_S = filter_sequences_by_ids(landing_sequences, allowed_ids_set)

        print("✅ sequences sizes:", len(takeoff_sequences_S), len(cruise_sequences_S), len(landing_sequences_S))

        # Safety: ensure we really got S
        assert len(takeoff_sequences_S) == S, f"takeoff_sequences_S has {len(takeoff_sequences_S)} != {S}"
        assert len(cruise_sequences_S)  == S, f"cruise_sequences_S has {len(cruise_sequences_S)} != {S}"
        assert len(landing_sequences_S) == S, f"landing_sequences_S has {len(landing_sequences_S)} != {S}"

        # 2) train the three phase models (your exact calls)
        model_takeoff, means_takeoff, stds_takeoff, errors_takeoff = train_and_eval_phase(takeoff_sequences_S, "takeoff")
        model_cruise,  means_cruise,  stds_cruise,  errors_cruise  = train_and_eval_phase(cruise_sequences_S,  "cruising")
        model_landing, means_landing, stds_landing, errors_landing = train_and_eval_phase(landing_sequences_S, "landing")
        lstm_results = get_mean_haversine_error(
            validation_flight_ids, 
            flights_data, 
            model_takeoff, 
            means_takeoff, 
            stds_takeoff,
            model_cruise, 
            means_cruise, 
            stds_cruise,
            model_landing, 
            means_landing, 
            stds_landing
        )
        print("📉 LSTM mean haversine (km):", lstm_results["LSTM"])

        # Save results incrementally (so if it crashes at S=400 you keep earlier results)
        upsert_result_record(
            results_obj,
            sample_size=S,
            payload={
                "batch_file": str(batch_file.as_posix()),
                "LSTM": lstm_results["LSTM"],
            },
        )
        save_json(RESULTS_PATH, results_obj)
        print(f"💾 Saved to {RESULTS_PATH}")

    print("\n✅ Done. Final results saved to:", RESULTS_PATH)


In [ ]:
run_lstm_loop_over_batches(
    flights_data=flights_data,
    takeoff_sequences=takeoff_sequences,
    cruise_sequences=cruise_sequences,
    landing_sequences=landing_sequences,
    validation_flight_ids=validation_flight_ids,
)